In [ ]:
import re
import ast
import json
import torch
from tqdm import tqdm
import pandas as pd
import numpy as np
from openai import OpenAI
from scipy.spatial.distance import cosine
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModel, AutoTokenizer
from sentence_transformers import SentenceTransformer, util
from sentence_transformers.util import cos_sim

## Create an AE mapping list with embeddings

In [ ]:
tirz_df = pd.read_excel('FAERS/Cleaned_Tirzepatide_AEs.xlsx')
sema_df = pd.read_excel('FAERS/Cleaned_Semaglutide_AEs.xlsx')
ozem_df = pd.read_excel('FAERS/Cleaned_Ozempic_AEs.xlsx')
rybe_df = pd.read_excel('FAERS/Cleaned_Rybelsus_AEs.xlsx')
wego_df = pd.read_excel('FAERS/Cleaned_Wegovy_AEs.xlsx')
zepb_df = pd.read_excel('FAERS/Cleaned_Zepbound_AEs.xlsx')
moun_df = pd.read_excel('FAERS/Cleaned_Mounjaro_AEs.xlsx')
lira_df = pd.read_excel('FAERS/Cleaned_Liraglutide_AEs.xlsx')
saxe_df = pd.read_excel('FAERS/Cleaned_Saxenda_AEs.xlsx')
vict_df = pd.read_excel('FAERS/Cleaned_Victoza_AEs.xlsx')

merged_df = pd.concat([tirz_df, sema_df, ozem_df, rybe_df, wego_df, zepb_df, moun_df, lira_df, saxe_df, vict_df], ignore_index=True)

merged_df = merged_df.drop_duplicates(subset=['AE', 'Reaction Group'])

merged_df.to_excel('FAERS/Merged_AEs.xlsx', index=False)

In [ ]:
merged_df.shape

In [ ]:
model = SentenceTransformer("pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb")
ae_texts = merged_df['AE'].dropna().astype(str).str.strip().tolist()

In [ ]:
embeddings = []
for ae in tqdm(ae_texts, desc="Embedding AE terms"):
    emb = model.encode(ae, convert_to_numpy=True, normalize_embeddings=True)  # normalize for cosine sim
    embeddings.append(emb)

In [ ]:
embedding_df = pd.DataFrame(embeddings, columns=[f'emb_{i}' for i in range(embeddings[0].shape[0])])
merged_df_clean = merged_df[merged_df['AE'].notna() & merged_df['AE'].str.strip().astype(bool)].reset_index(drop=True)
merged_df_embedded = pd.concat([merged_df_clean, embedding_df], axis=1)

In [ ]:
merged_df_embedded

In [ ]:
merged_df_embedded.to_csv("FAERS/Merged_AEs_with_SentenceBioBERT.csv", index=False)

## Match side effects with AEs

In [ ]:
review_df = pd.read_csv("WebMD/combined_extracted_reviews.csv")
merged_df_embedded = pd.read_csv("FAERS/Merged_AEs_with_SentenceBioBERT.csv")

In [ ]:
review_df['structured_info'] = review_df['structured_info'].apply(ast.literal_eval)

side_effects = []
for row in review_df['structured_info']:
    for item in row.get('side_effects', []):
        name = item.get('name')
        if name:
            side_effects.append(name.strip())

unique_side_effects = list(set(side_effects))

In [ ]:
unique_side_effects

In [ ]:
ae_texts = merged_df_embedded['AE'].tolist()
ae_embeddings = merged_df_embedded[[f'emb_{i}' for i in range(768)]].dropna().values

In [ ]:
results = []
threshold = 0

for effect in unique_side_effects:
    try:
        query_emb = model.encode(effect, convert_to_numpy=True, normalize_embeddings=True)

        similarities = np.dot(ae_embeddings, query_emb)

        top_indices = similarities.argsort()[::-1]
        top_matches = [(ae_texts[i], similarities[i]) for i in top_indices if similarities[i] >= threshold]
        top_matches = top_matches[:10]

        results.append({
            'Extracted Side Effect': effect,
            'Top Matches': top_matches
        })
    except Exception as e:
        results.append({
            'Extracted Side Effect': effect,
            'Top Matches': f"Error: {e}"
        })

In [ ]:
for res in results[:20]:
    print(f"\nOriginal: {res['Extracted Side Effect']}")
    if isinstance(res['Top Matches'], str):
        print(f"  {res['Top Matches']}")
    elif not res['Top Matches']:
        print("  No match with similarity > 0.4")
    else:
        for match, score in res['Top Matches']:
            print(f"  Match: {match} (Similarity: {score:.4f})")